In [ ]:
job_id = str(uuid.uuid4())
job_name = 'DM_DIM_CUS_CUSTODIES_LOAD'
layer_name = 'DATAMART'
start_time = datetime.now()
rows_loaded = 0
rows_failed = 0
run_status = 'SUCCESS'
error_message = None

try:
    # Step 1: Create temp table from source
    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_DM_CUS_CUSTODIES AS
    SELECT DISTINCT
        CUS_ID,
        LEGAL_STATUS_CODE_AV_ID,
        LEGAL_STATUS_CODE_DESC,
        CUSTODY_TYPE_AV_ID,
        CUSTODY_TYPE_DESC,
        CUSTODY_START_DATE,
        CUSTODY_END_DATE,
        SHA2(CONCAT_WS('|',
            COALESCE(TO_VARCHAR(CUS_ID), ''),
            COALESCE(LEGAL_STATUS_CODE_AV_ID, ''),
            COALESCE(LEGAL_STATUS_CODE_DESC, ''),
            COALESCE(CUSTODY_TYPE_AV_ID, ''),
            COALESCE(CUSTODY_TYPE_DESC, ''),
            COALESCE(TO_VARCHAR(CUSTODY_START_DATE), ''),
            COALESCE(TO_VARCHAR(CUSTODY_END_DATE), '')
        ), 256) AS CHECKSUM
    FROM {DB}.{DATAMART}.{DM_SOURCE}
    WHERE CUS_ID IS NOT NULL
    """).collect()

    # Step 2: SCD2 - Expire old records where checksum changed
    session.sql(f"""
    UPDATE {DB}.{DATAMART}.DIM_DM_CUS_CUSTODIES_DT tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_DM_CUS_CUSTODIES src
    WHERE tgt.CUS_ID = src.CUS_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    # Step 3: SCD2 - Insert new/changed versions
    insert_result = session.sql(f"""
    INSERT INTO {DB}.{DATAMART}.DIM_DM_CUS_CUSTODIES_DT (
        CUS_ID, LEGAL_STATUS_CODE_AV_ID, LEGAL_STATUS_CODE_DESC,
        CUSTODY_TYPE_AV_ID, CUSTODY_TYPE_DESC,
        CUSTODY_START_DATE, CUSTODY_END_DATE,
        CREATED_DATE, UPDATED_DATE, CHECKSUM
    )
    SELECT
        src.CUS_ID, src.LEGAL_STATUS_CODE_AV_ID, src.LEGAL_STATUS_CODE_DESC,
        src.CUSTODY_TYPE_AV_ID, src.CUSTODY_TYPE_DESC,
        src.CUSTODY_START_DATE, src.CUSTODY_END_DATE,
        CURRENT_TIMESTAMP(), NULL, src.CHECKSUM
    FROM TEMP_DM_CUS_CUSTODIES src
    WHERE NOT EXISTS (
        SELECT 1 FROM {DB}.{DATAMART}.DIM_DM_CUS_CUSTODIES_DT tgt
        WHERE tgt.CUS_ID = src.CUS_ID
          AND tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), rows_loaded, rows_failed, run_status, None)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'DIM_DM_CUS_CUSTODIES loaded. Rows: {rows_loaded}')
    print(f'SUCCESS | Rows Loaded: {rows_loaded}')

except Exception as error:
    run_status = 'FAILED'
    rows_failed = 1
    error_message = str(error)
    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), 0, rows_failed, run_status, error_message)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'DIM_DM_CUS_CUSTODIES failed: {error_message}')
    print(f'FAILED: {error_message}')